## Accelerate Inference: Neural Network Pruning

In [2]:
import os
import numpy as np
import cv2
import pickle
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# untar
!ls
!tar -xvzf dataset.tar.gz
# load train
train_images = pickle.load(open('/content/drive/MyDrive/605/dataset/train_images.pkl', 'rb'))
train_labels = pickle.load(open('/content/drive/MyDrive/605/dataset/train_labels.pkl', 'rb'))
# load val
val_images = pickle.load(open('/content/drive/MyDrive/605/dataset/val_images.pkl', 'rb'))
val_labels = pickle.load(open('/content/drive/MyDrive/605/dataset/val_labels.pkl', 'rb'))

drive  sample_data
tar (child): dataset.tar.gz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now


In [6]:
train_images = torch.tensor(train_images, dtype=torch.float32)
val_images = torch.tensor(val_images, dtype=torch.float32)

train_images = train_images.permute(0, 3, 1, 2)
val_images = val_images.permute(0, 3, 1, 2)

In [7]:
train_dataset = TensorDataset(train_images,
                              torch.tensor(train_labels.squeeze(), dtype=torch.long))
val_dataset = TensorDataset(val_images,
                            torch.tensor(val_labels.squeeze(), dtype=torch.long))

In [8]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [9]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()

        self.model = nn.Sequential(
            # First block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Second block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Flatten layer
            nn.Flatten(),

            # Fully connected block: Dense -> ReLU -> Dropout -> Dense -> Softmax
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 5),
        )

    def forward(self, x):
        return self.model(x)

In [10]:
model = ConvNet()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)

In [11]:
model = model.to(device)
summary(model, input_size=(3, 25, 25))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 25, 25]             896
              ReLU-2           [-1, 32, 25, 25]               0
            Conv2d-3           [-1, 32, 23, 23]           9,248
              ReLU-4           [-1, 32, 23, 23]               0
         MaxPool2d-5           [-1, 32, 11, 11]               0
           Dropout-6           [-1, 32, 11, 11]               0
            Conv2d-7           [-1, 64, 11, 11]          18,496
              ReLU-8           [-1, 64, 11, 11]               0
            Conv2d-9             [-1, 64, 9, 9]          36,928
             ReLU-10             [-1, 64, 9, 9]               0
        MaxPool2d-11             [-1, 64, 4, 4]               0
          Dropout-12             [-1, 64, 4, 4]               0
          Flatten-13                 [-1, 1024]               0
           Linear-14                  [

In [12]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the training loop
    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()  # Zero the parameter gradients
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Update tqdm description with current loss and accuracy
        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy

In [13]:
def validate(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the validation loop
    val_loader_tqdm = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():  # Disable gradient calculations for validation
        for inputs, labels in val_loader_tqdm:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Track loss and accuracy
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Update tqdm description with current validation loss and accuracy
            val_loader_tqdm.set_postfix(loss=val_loss / total, accuracy=100 * correct / total)

    val_accuracy = 100 * correct / total
    val_loss = val_loss / len(val_loader)
    return val_loss, val_accuracy

In [14]:
# Main training loop
num_epochs = 50
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    # Training
    train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # Validation
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)

    # Print epoch results
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')

Epoch 1/50


Epoch [1/50], Train Loss: 1.5199, Train Acc: 30.12%, Val Loss: 1.3920, Val Acc: 39.52%
Epoch 2/50


Epoch [2/50], Train Loss: 1.3840, Train Acc: 39.85%, Val Loss: 1.3252, Val Acc: 42.61%
Epoch 3/50


Epoch [3/50], Train Loss: 1.3289, Train Acc: 42.84%, Val Loss: 1.2731, Val Acc: 45.86%
Epoch 4/50


Epoch [4/50], Train Loss: 1.2921, Train Acc: 44.90%, Val Loss: 1.2476, Val Acc: 47.01%
Epoch 5/50


...

Epoch [46/50], Train Loss: 0.7399, Train Acc: 71.31%, Val Loss: 0.8497, Val Acc: 65.78%
Epoch 47/50


Epoch [47/50], Train Loss: 0.7257, Train Acc: 72.30%, Val Loss: 0.8487, Val Acc: 65.62%
Epoch 48/50


Epoch [48/50], Train Loss: 0.7191, Train Acc: 72.69%, Val Loss: 0.8286, Val Acc: 67.21%
Epoch 49/50


Epoch [49/50], Train Loss: 0.7061, Train Acc: 73.28%, Val Loss: 0.9228, Val Acc: 64.24%
Epoch 50/50


Epoch [50/50], Train Loss: 0.6994, Train Acc: 73.49%, Val Loss: 0.8281, Val Acc: 66.93%


In [15]:
torch.save(model.state_dict(), 'my_model_weights_first.pt', _use_new_zipfile_serialization=False)

### Zeroing weights

Example on how to set weights to zero. You should determine which weights to set to zero using the approaches you defined in the form.

In [16]:
# Set weights that are less than 0.5 to zero
with torch.no_grad():  # disable gradient tracking for efficiency
    for name, param in model.named_parameters():
        if "weight" in name:  # only apply to weights, skip biases
            param[param < 0.9] = 0

In [17]:
val_loss, val_accuracy = validate(model, val_loader, criterion, device)

In [18]:
val_accuracy

19.247524752475247

In [19]:
torch.save(model.state_dict(), 'my_model_weights_second.pt', _use_new_zipfile_serialization=False)

In [104]:
def model_zero_weights(model):
    zeros, weights = 0, 0

    for name, param in model.named_parameters():
        if 'weight' not in name:
            continue
        zeros += (param == 0).sum().item()
        weights += param.numel()

    sparsity_level = zeros / weights

    print(f"Sparsity: {sparsity_level:.2%} (Zero weights: {zeros}/{weights})")
    return sparsity_level

In [105]:
def optimize_model(model, train_data, val_data, num_epochs=5):
    """
    Optimizes the model through training and validation cycles.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    best_acc = 0.0

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = train_one_epoch(model, train_data, optimizer, criterion, device)

        # Validation phase
        model.eval()
        val_loss, val_acc = validate(model, val_data, criterion, device)

        # Track best accuracy
        if val_acc > best_acc:
            best_acc = val_acc

        print(f"Fine-tuning Epoch {epoch+1}/{num_epochs}: Val Acc: {best_acc:.2f}%")


    return model

In [106]:
def l1_norm_prune_methods(model, pruning_rate=0.2):
    model.eval()

    for layer in model.modules():
        if not isinstance(layer, (nn.Conv2d, nn.Linear)): #only do Conv2d and Linear
            continue

        weight = layer.weight.data
        if weight.dim() == 4:  # Conv2d weights
            norms = weight.abs().sum(dim=(1,2,3))
        else:  # Linear weights
            norms = weight.abs().sum(dim=1)

        cutoff_rank = int(pruning_rate * norms.numel())
        if cutoff_rank == 0:
            continue

        #threshold val calculation using torch.kthvalue
        threshold = torch.kthvalue(norms, cutoff_rank).values

        #using threshold to create mask and zero them out
        with torch.no_grad():
            mask = norms > threshold
            zero_indices = ~mask
            weight[zero_indices] = 0

    return model

In [107]:
def iterative_pruning(model, train_data, val_data, target_sparsity=0.3, n_iters=3):
    for i in range(n_iters):
        model = l1_norm_prune_methods(model, pruning_rate=0.1)  # prune 10% each time
        model = optimize_model(model, train_data, val_data, num_epochs=3)

        sparsity = model_zero_weights(model)
        if sparsity >= target_sparsity:
            break
    return model

In [111]:
def test_pruning_rates():
    model = ConvNet().to(device)
    model.load_state_dict(torch.load('my_model_weights_first.pt'))

    # try different pruning rates here
    pruning_rates = [0.03, 0.05,0.08, 0.1,0.12, 0.15, 0.18, 0.2, 0.3, 0.32, 0.35, 0.36, 0.38, 0.4]

    performance_records = []

    #requirements below as mentioned in guidance
    min_accuracy = 60
    min_score = 0.36

    for rate in pruning_rates:
        print(f"\n===== Pruning Rate: {rate:.0%} =====")
        pruned_model = ConvNet().to(device)
        pruned_model.load_state_dict(model.state_dict())

        pruned_model_prep = l1_norm_prune_methods(pruned_model, rate)
        pruned_model = optimize_model(pruned_model_prep, train_loader, val_loader, num_epochs=5)

        val_loss, val_acc = validate(pruned_model, val_loader, criterion, device)
        sparsity = model_zero_weights(pruned_model)

        #calculation according to guidance
        score = (val_acc/100 + sparsity) / 2

        #only save model that meets all requirements
        if val_acc > min_accuracy and score > min_score:
            torch.save(pruned_model.state_dict(), f'my_model_weights_test_{rate}.pt', _use_new_zipfile_serialization=False)
            print(f"Saved valid model (Score: {score:.4f})")
        else:
            print(f"Model invalid (Score: {score:.4f})")

        performance_records.append({'rate': rate, 'acc': val_acc, 'sparsity': sparsity, 'score': score})


    print("\nFinal Results:")
    print("Rate | Acc  | Sparsity | Score")
    for r in performance_records:
        print(f"{r['rate']:.0%}   | {r['acc']:.2f}% | {r['sparsity']:.2%} | {r['score']:.4f}")


test_pruning_rates()

<ipython-input-111-915524d7fa6c>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('my_model_weights_first.pt'))



===== Pruning Rate: 3% =====


Fine-tuning Epoch 1/5: Val Acc: 67.13%


Fine-tuning Epoch 2/5: Val Acc: 67.13%


Fine-tuning Epoch 3/5: Val Acc: 67.45%


...

Fine-tuning Epoch 2/5: Val Acc: 53.74%


Fine-tuning Epoch 3/5: Val Acc: 56.12%


Fine-tuning Epoch 4/5: Val Acc: 56.59%


Fine-tuning Epoch 5/5: Val Acc: 58.85%


Sparsity: 23.35% (Zero weights: 138300/592224)
Model invalid (Score: 0.4110)

Final Results:
Rate | Acc  | Sparsity | Score
3%   | 67.33% | 2.70% | 0.3501
5%   | 67.29% | 4.68% | 0.3598
8%   | 67.64% | 6.71% | 0.3718
10%   | 66.65% | 8.35% | 0.3750
12%   | 67.41% | 10.08% | 0.3874
15%   | 67.45% | 11.81% | 0.3963
18%   | 66.57% | 13.18% | 0.3987
20%   | 65.31% | 14.39% | 0.3985
30%   | 62.73% | 18.63% | 0.4068
32%   | 61.94% | 19.58% | 0.4076
35%   | 61.11% | 21.60% | 0.4135
36%   | 60.48% | 21.93% | 0.4120
38%   | 60.55% | 22.79% | 0.4167
40%   | 58.85% | 23.35% | 0.4110
